# Diseño del Pipeline de Datos y Prompting
En este notebook se diseña el pipeline que conecta los datos preprocesados 
con el modelo generativo Gemini Flash para generar crónicas periodísticas automáticas.

In [13]:
import pandas as pd
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

df = pd.read_csv("../data/processed/epl_clean.csv")
print(f"Dataset cargado: {df.shape[0]} partidos")

Dataset cargado: 9325 partidos


## Construcción del prompt
Se diseña una función que traduce las estadísticas de un partido en un prompt 
estructurado para guiar al modelo en la generación de la crónica.

In [14]:
def build_prompt(row):
    resultado_map = {'H': f"ganó {row['HomeTeam']}", 'A': f"ganó {row['AwayTeam']}", 'D': "empate"}
    resultado_ht_map = {'H': f"ganaba {row['HomeTeam']}", 'A': f"ganaba {row['AwayTeam']}", 'D': "iban empatados"}

    favorito = row['Favorite']
    sorpresa = "Sí, fue una sorpresa" if row['Upset'] else "No, ganó el favorito o empató"

    prompt = f"""
Eres un periodista deportivo experto en fútbol inglés. 
Redactá una crónica periodística en español de aproximadamente 200 palabras 
sobre el siguiente partido de la Premier League.

La crónica debe tener un título atractivo, narrar el desarrollo del partido 
y destacar los datos más relevantes. Usá un tono dinámico y profesional.

--- DATOS DEL PARTIDO ---
Fecha: {row['MatchDate']}
Local: {row['HomeTeam']}
Visitante: {row['AwayTeam']}
Resultado final: {int(row['FTHome'])} - {int(row['FTAway'])} ({resultado_map[row['FTResult']]})
Al descanso: {int(row['HTHome'])} - {int(row['HTAway'])} ({resultado_ht_map[row['HTResult']]})

Estadísticas:
- Tiros (local / visitante): {int(row['HomeShots'])} / {int(row['AwayShots'])}
- Tiros al arco (local / visitante): {int(row['HomeTarget'])} / {int(row['AwayTarget'])}
- Corners (local / visitante): {int(row['HomeCorners'])} / {int(row['AwayCorners'])}
- Faltas (local / visitante): {int(row['HomeFouls'])} / {int(row['AwayFouls'])}
- Tarjetas amarillas (local / visitante): {int(row['HomeYellow'])} / {int(row['AwayYellow'])}
- Tarjetas rojas (local / visitante): {int(row['HomeRed'])} / {int(row['AwayRed'])}

Contexto:
- Favorito según cuotas: {favorito}
- ¿Fue sorpresa el resultado?: {sorpresa}
- Forma reciente local (últimos 5 partidos, puntos): {row['Form5Home']}
- Forma reciente visitante (últimos 5 partidos, puntos): {row['Form5Away']}
- Diferencia de Elo (local - visitante): {row['EloDiff']} puntos
"""
    return prompt

## Prueba con un partido
Generamos la crónica de un partido de prueba para validar el pipeline.

In [15]:
partido = df.iloc[100]
prompt = build_prompt(partido)

print("=== PROMPT ENVIADO ===")
print(prompt)

=== PROMPT ENVIADO ===

Eres un periodista deportivo experto en fútbol inglés. 
Redactá una crónica periodística en español de aproximadamente 200 palabras 
sobre el siguiente partido de la Premier League.

La crónica debe tener un título atractivo, narrar el desarrollo del partido 
y destacar los datos más relevantes. Usá un tono dinámico y profesional.

--- DATOS DEL PARTIDO ---
Fecha: 2000-10-28
Local: Aston Villa
Visitante: Charlton
Resultado final: 2 - 1 (ganó Aston Villa)
Al descanso: 2 - 0 (ganaba Aston Villa)

Estadísticas:
- Tiros (local / visitante): 12 / 7
- Tiros al arco (local / visitante): 7 / 4
- Corners (local / visitante): 9 / 6
- Faltas (local / visitante): 13 / 11
- Tarjetas amarillas (local / visitante): 3 / 2
- Tarjetas rojas (local / visitante): 0 / 0

Contexto:
- Favorito según cuotas: Aston Villa
- ¿Fue sorpresa el resultado?: No, ganó el favorito o empató
- Forma reciente local (últimos 5 partidos, puntos): 8.0
- Forma reciente visitante (últimos 5 partidos, pu

In [16]:
def generate_cronica(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500
    )
    return response.choices[0].message.content

partido = df.iloc[100]
prompt = build_prompt(partido)
cronica = generate_cronica(prompt)

print("=== CRÓNICA GENERADA ===")
print(cronica)

=== CRÓNICA GENERADA ===
**Aston Villa se impone con autoridad sobre Charlton**

En un partido emocionante en la Premier League, Aston Villa logró una victoria merecida sobre Charlton con un resultado final de 2-1. Desde el inicio, los locales demostraron su superioridad, llevándose la iniciativa y controlando el ritmo del juego. Al descanso, ya se encontraban 2-0 arriba, gracias a su destacada eficacia en ataque.

Las estadísticas del partido reflejan la dominancia de Aston Villa, con 12 tiros totales y 7 al arco, en comparación con los 7 tiros y 4 al arco de Charlton. Además, los locales tuvieron 9 corners, superando los 6 del equipo visitante. A pesar de que Charlton intentó reaccionar en la segunda mitad, solo pudo anotar un gol, lo que no fue suficiente para cambiar el curso del encuentro.

Con esta victoria, Aston Villa consolida su posición en la tabla, mientras que Charlton deberá seguir trabajando para mejorar su rendimiento. La diferencia de Elo de 114.4 puntos entre ambos eq